# 04 – Baseline: BoW / TF-IDF × Naive Bayes / Logistic Regression (Mốc 1)

Mỗi mô hình báo **hai con số**: F1-macro trên ViSD4SA test (trong miền của A) và F1-macro trên tập vàng Shopee.

⚠️ Nếu chưa có `labeling/gold_500.csv`, notebook **tạm** chấm trên cả kho vàng (746 câu, 10 sản phẩm) bằng **nhãn sao**. Con số đó chỉ để tham khảo, không phải kết quả cuối. Gán nhãn xong thì chạy lại toàn bộ notebook.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report

P = "../data/processed"
A = pd.read_csv(f"{P}/visd_train.csv")[["text_prep", "label"]]
A_test = pd.read_csv(f"{P}/visd_test.csv")[["text_prep", "label"]]
B = pd.read_csv(f"{P}/shopee_train_relabel.csv").rename(columns={"weak_label": "label"})[["text_prep", "label"]]
Aug = pd.read_csv(f"{P}/aug_train.csv")[["text_prep", "weak_label"]].rename(columns={"weak_label": "label"})   # file đã có cột label gốc (positive/negative)
pool = pd.read_csv(f"{P}/shopee_gold_pool.csv")

GOLD = "../labeling/gold_500.csv"
if os.path.exists(GOLD):
    g = pd.read_csv(GOLD)[["id", "sentiment"]].dropna()
    g["sentiment"] = g.sentiment.str.strip().str.lower()
    gold = g[g.sentiment.isin(["pos", "neg", "neu"])].merge(pool[["id", "text_prep"]], on="id")
    GOLD_NAME = f"tập vàng (nhãn tay, {len(gold)} câu)"
else:
    gold = pool.rename(columns={"weak_label": "sentiment"})[["text_prep", "sentiment"]]
    GOLD_NAME = f"TẠM: kho vàng nhãn sao ({len(gold)} câu)"
print("Đánh giá ngoài miền trên:", GOLD_NAME)
print(gold.sentiment.value_counts())

train_sets = {"A": A, "B": B, "A+B": pd.concat([A, B], ignore_index=True), "A+B+Aug": pd.concat([A, B, Aug], ignore_index=True)}
for k, d in train_sets.items():
    print(f"{k:8s}", len(d), dict(d.label.value_counts()))

Đánh giá ngoài miền trên: tập vàng (nhãn tay, 300 câu)
sentiment
pos    148
neg    121
neu     31
Name: count, dtype: int64
A        7344 {'pos': np.int64(4596), 'neg': np.int64(2440), 'neu': np.int64(308)}
B        1788 {'pos': np.int64(1153), 'neg': np.int64(591), 'neu': np.int64(44)}
A+B      9132 {'pos': np.int64(5749), 'neg': np.int64(3031), 'neu': np.int64(352)}
A+B+Aug  10337 {'pos': np.int64(6401), 'neg': np.int64(3584), 'neu': np.int64(352)}


In [2]:
vecs = {"BoW": lambda: CountVectorizer(ngram_range=(1, 2), min_df=2),
        "TF-IDF": lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)}
clfs = {"NB": lambda: MultinomialNB(),
        "LR": lambda: LogisticRegression(max_iter=2000, class_weight="balanced")}

rows, models = [], {}
for cfg, d in train_sets.items():
    for vn, v in vecs.items():
        for cn, c in clfs.items():
            m = make_pipeline(v(), c()).fit(d.text_prep, d.label)
            models[(cfg, f"{vn}+{cn}")] = m
            rows.append({"train": cfg, "model": f"{vn}+{cn}",
                         "F1_ViSD_test": f1_score(A_test.label, m.predict(A_test.text_prep), average="macro"),
                         "F1_gold": f1_score(gold.sentiment, m.predict(gold.text_prep), average="macro")})
res = pd.DataFrame(rows)
res.to_csv("../report/baseline_results.csv", index=False)
print("Ngoài miền:", GOLD_NAME)
display(res.pivot(index="model", columns="train", values="F1_gold").round(3))
print("Trong miền: ViSD4SA test")
display(res.pivot(index="model", columns="train", values="F1_ViSD_test").round(3))

Ngoài miền: tập vàng (nhãn tay, 300 câu)


train,A,A+B,A+B+Aug,B
model,,,,
BoW+LR,0.571,0.570,0.584,0.574
BoW+NB,0.561,0.583,0.558,0.607
TF-IDF+LR,0.659,0.639,0.659,0.605
TF-IDF+NB,0.388,0.464,0.431,0.536


Trong miền: ViSD4SA test


train,A,A+B,A+B+Aug,B
model,,,,
BoW+LR,0.738,0.724,0.727,0.539
BoW+NB,0.586,0.585,0.578,0.465
TF-IDF+LR,0.708,0.710,0.711,0.509
TF-IDF+NB,0.543,0.554,0.535,0.528


In [3]:
# Báo cáo chi tiết từng lớp cho mô hình tốt nhất trên tập vàng
best = res.loc[res.F1_gold.idxmax()]
print("Tốt nhất:", best.train, best.model, f"F1_gold = {best.F1_gold:.3f}")
m = models[(best.train, best.model)]
print(classification_report(gold.sentiment, m.predict(gold.text_prep), digits=3))

Tốt nhất: A+B+Aug TF-IDF+LR F1_gold = 0.659
              precision    recall  f1-score   support

         neg      0.843     0.843     0.843       121
         neu      0.625     0.161     0.256        31
         pos      0.819     0.946     0.878       148

    accuracy                          0.823       300
   macro avg      0.762     0.650     0.659       300
weighted avg      0.808     0.823     0.800       300



### Nhận xét về nguy cơ học lối tắt

Trong kết quả train B, các từ chỉ sản phẩm như “quần”, “sạc” có trọng số cao ở lớp tiêu cực. Điều này gợi ý cần kiểm tra phụ thuộc vào cơ cấu sản phẩm, chưa chứng minh mô hình chỉ học lối tắt. Với A+B, các tín hiệu như “không”, “tệ”, “thất_vọng” nổi bật hơn; “pin trâu” ở lớp tích cực vẫn cho thấy dấu vết miền điện thoại. Vì vậy cần chia theo sản phẩm, đọc lỗi trên mặt hàng mới và tránh diễn giải trọng số như quan hệ nhân quả.


In [4]:
for cfg in ["B", "A+B"]:
    pipe = models[(cfg, "TF-IDF+LR")]
    vocab = np.array(pipe[0].get_feature_names_out())
    lr = pipe[1]
    print(f"===== train = {cfg} =====")
    for i, cls in enumerate(lr.classes_):
        top = vocab[np.argsort(lr.coef_[i])[::-1][:20]]
        print(f"{cls}: {', '.join(top)}")
    print()

===== train = B =====
neg: không, quá, kém, như, lại, đặt, mà, tệ, thất_vọng, quần, không đúng, cửa_hàng, hết, bị, rách, bung, sạc, lỗi, sai, mấy
neu: hơi, tạm, có, nóng, luong, thì, nhiều chỉ, biết, chỉ, chổ, của đấy, lùi, nhẹ, đúg, biết có, màu thì, cái, có_vẻ, tạm được, được không
pos: tốt, nhanh, ổn, giao hàng, đúng, hàng, rất, sản_phẩm, giá, rẻ, đẹp, dùng, hàng nhanh, ủng_hộ, chất_lượng, mùi, giao nhanh, da, sử_dụng, nên

===== train = A+B =====
neg: không, tệ, thất_vọng, kém, chán, mà, bị, đơ, thua, đổi, lỗi, không nên, đặt, chậm, như, khi, quá, tắt, ra, cửa_hàng
neu: tạm, tạm ổn, bình_thường, cũng, cũng ổn, tạm được, được, ổn nhưng, hơi, dùng cũng, cũng được, tương_đối, cũng tạm, ổn, máy bình_thường, nhưng, nói_chung tạm, hơn, luong, hoặc
pos: ổn, tốt, rất, đẹp, tuyệt_vời, ngon, mượt, trâu, chê, sản_phẩm, không bị, mọi, mọi thứ, đều ổn, rẻ, nhiệt_tình, rất ổn, thứ, hàng, pin trâu

